In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


**Tabla: Customers**

In [46]:
df_customers = pd.read_csv("Raw_Data/customers.csv")
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        5000 non-null   str    
 1   full_name          4894 non-null   str    
 2   age                4814 non-null   float64
 3   gender             4888 non-null   str    
 4   email              4889 non-null   str    
 5   phone              4814 non-null   str    
 6   street_address     4823 non-null   str    
 7   city               4903 non-null   str    
 8   state              4907 non-null   str    
 9   zip_code           4904 non-null   float64
 10  registration_date  5000 non-null   str    
 11  preferred_channel  4886 non-null   str    
dtypes: float64(2), str(10)
memory usage: 468.9 KB


*Verificamos la cantidad de nulos*

In [47]:
df_customers.isnull().sum().loc[lambda x: x > 0]

full_name            106
age                  186
gender               112
email                111
phone                186
street_address       177
city                  97
state                 93
zip_code              96
preferred_channel    114
dtype: int64

*Verificamos duplicados*

In [48]:
duplicados_customers = df_customers.duplicated().sum()
print(duplicados_customers)

0


**Vamos a comenzar la limpieza creando un diccionario para los nulos. Todos exceptos los de edad para evitar problemas.**

In [49]:
reglas_limpieza = {
    'full_name': 'Nombre No Registrado',
    'gender': 'Sin Especificar',
    'email': 'Sin Especificar',
    'phone': 'Sin Especificar',
    'preferred_channel': 'Sin Especificar',
    'street_address': 'Desconocido',
    'city': 'Desconocido',
    'state': 'Desconocido',
    'zip_code': 'Desconocido'
}

#Aplicamos todas las reglas de una sola vez
df_customers = df_customers.fillna(reglas_limpieza)

*Ahora aplicamos reglas de texto en los datos para evitar problemas con la integración en SQL*

In [50]:
df_customers['full_name'] = df_customers['full_name'].str.strip().str.title()

*Verificamos que todo haya quedado a 0 (excepto 'age', que la dejamos como nulo a propósito)*

In [51]:
print(df_customers.isnull().sum())

customer_id            0
full_name              0
age                  186
gender                 0
email                  0
phone                  0
street_address         0
city                   0
state                  0
zip_code               0
registration_date      0
preferred_channel      0
dtype: int64


*Ahora que la base está limpia de nulos, podemos comenzar con su integración para corregir los errores de la otra base. Sin embargo, aún nos falta otra tabla*

In [ ]:

df_customers.to_csv('customers_limpio.csv', index=False)

print("¡Archivo customers_limpio.csv generado con éxito!")

¡Archivo customers_limpio.csv generado con éxito!


**Tabla: transactions**

*Para los demas nulos, necesitamos los datos provenientes de esta tabla*

In [38]:
df_transactions = pd.read_csv(r'A:\Escritorio\Proyectos DATA\CRM\end-to-end-saas-pipeline\Raw_Data\transactions.csv')

df_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 32295 entries, 0 to 32294
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    32295 non-null  str    
 1   customer_id       32295 non-null  str    
 2   product_name      31617 non-null  str    
 3   product_category  31609 non-null  str    
 4   quantity          31651 non-null  float64
 5   price             31673 non-null  float64
 6   transaction_date  32295 non-null  str    
 7   store_location    31651 non-null  str    
 8   payment_method    31635 non-null  str    
 9   discount_applied  31684 non-null  float64
dtypes: float64(3), str(7)
memory usage: 2.5 MB


In [39]:
df_transactions.isnull().sum().loc[lambda x: x > 0]

product_name        678
product_category    686
quantity            644
price               622
store_location      644
payment_method      660
discount_applied    611
dtype: int64

In [40]:
reglas_limpieza_transactions = {'product_name':"Uknow",    
'product_category': "Uknow",             
'store_location':'Uknow',      
'payment_method':"Uknow",    
'discount_applied': 0.0  # Asumimos que nulo = no hay descuento    
    
}

In [41]:
df_transactions = df_transactions.fillna(reglas_limpieza_transactions)

*Aquí hay un asuntito.... Para evitar alterar el analisis, borramos los datos de precios y cantidades que sean nulos*

In [42]:
nulos_transactions_precio_quantity = df_transactions[['price','quantity']].isnull().sum().sum()
print(nulos_transactions_precio_quantity)

1266


In [44]:
df_transactions = df_transactions.dropna(subset=['price', 'quantity'])
df_transactions.isnull().sum()

transaction_id      0
customer_id         0
product_name        0
product_category    0
quantity            0
price               0
transaction_date    0
store_location      0
payment_method      0
discount_applied    0
dtype: int64

*Ahora que la base está limpia, la generamos y pasamos a utilizarla en nuestro analisis principal*

In [52]:

df_transactions.to_csv('transactions_limpio.csv', index=False)

print("¡Archivo transactions_limpio.csv generado con éxito!")

¡Archivo transactions_limpio.csv generado con éxito!
